In [1]:
import pandas as pd
from pathlib import Path

# Ensure data directories and paths are established
data_dir = Path("data/raw")
file_path_1 = data_dir / "customer_orders.parquet"
file_path_2 = data_dir / "pickup_point_orders.parquet"
file_path_3 = data_dir / "product_reference.parquet"
file_path_4 = data_dir / "product_snapshots.parquet"
file_path_5 = data_dir / "product_top.parquet"

# Load datasets
df_cust_ord = pd.read_parquet(file_path_1)
df_pup_ord = pd.read_parquet(file_path_2)
df_prod_ref = pd.read_parquet(file_path_3)
df_prod_snap = pd.read_parquet(file_path_4)
df_prod_top = pd.read_parquet(file_path_5)
df_pickup = pd.read_parquet(file_path_2)


In [2]:
# Get unique SKU-EAN combinations from snapshots
df_sku_ean_snap = (
    df_prod_snap[["sku", "ean_code"]]
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
)

# Count distinct EANs associated with each SKU
ean_counts = (
    df_sku_ean_snap
    .groupby("sku")["ean_code"]
    .nunique()
)

# Keep SKUs with exactly one distinct EAN
valid_skus = ean_counts[ean_counts == 1].index

# Build the mapping expected by your function
snapshot_mapping = (
    df_sku_ean_snap[
        df_sku_ean_snap["sku"].isin(valid_skus)
    ]
    .rename(columns={"ean_code": "snapshot_ean_code"})
    .reset_index(drop=True)
)

print(f"Total SKUs in snapshots: {len(ean_counts)}")
print(f"SKUs with exactly one EAN: {len(snapshot_mapping)}")
print(f"SKUs with multiple EANs: {(ean_counts > 1).sum()}")

display(snapshot_mapping.head())

Total SKUs in snapshots: 1597714
SKUs with exactly one EAN: 1597703
SKUs with multiple EANs: 11


,sku,snapshot_ean_code
0,003ed021-227b-49d9-b66a-3e0fab41a494,3160921965868
1,007f596a-38c9-45cf-be61-af83cbf7dfbc,5711953145506
2,02d8ea61-5bbe-45ab-b36d-4f410de04e7d,5411188096764
3,0352d7ef-a8dd-4830-bab9-3be5c967a0ab,5709345004692
4,03bc0299-a5be-4bf9-af67-5a43cc0ca005,8013356092065


In [12]:
# Define the known error period
ERROR_START_DATE = "2024-04-01"
ERROR_END_DATE = "2024-07-31"
# ERROR_START_DATE = "2000-01-01"
# ERROR_END_DATE = "2100-07-31"
# Make a copy
df_preview = df_pickup.copy()

df_preview["order_date"] = pd.to_datetime(
    df_preview["order_date"]
)

# Attach the snapshot EAN
comparison = df_preview.merge(
    snapshot_mapping,
    on="sku",
    how="left",
    validate="many_to_one",
    sort=False
)

# Reproduce the correction conditions
in_error_period = comparison["order_date"].between(
    ERROR_START_DATE,
    ERROR_END_DATE
)

has_reference = comparison["snapshot_ean_code"].notna()

mismatch = (
    comparison["ean_code"] != comparison["snapshot_ean_code"]
).fillna(False)

correction_mask = (
    in_error_period
    & has_reference
    & mismatch
)

# Extract rows that would be corrected
df_affected = comparison.loc[
    correction_mask
].copy()

df_affected = df_affected.rename(
    columns={
        "ean_code": "old_ean",
        "snapshot_ean_code": "corrected_ean"
    }
)

print(f"Rows that would be corrected: {len(df_affected)}")
print(f"Expected corrections: 29")
print(f"Matches expected count: {len(df_affected) == 29}")

display(
    df_affected.sort_values(["order_date", "sku"])
)

Rows that would be corrected: 48
Expected corrections: 29
Matches expected count: False


,order_date,pickup_point,sku,old_ean,quantity_ordered,new_price,n_orders,corrected_ean
317740,2024-04-25,2542941672fd21b8,3cf2950b-da59-44ee-b5e9-d5d86aba06e5,5411761258817,2,3.30,1,5400119547581
334586,2024-04-30,b73fc63876e68b0b,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3487400006769,1,1.20,1,8719200263796
335200,2024-04-30,cbcb62162e461101,d36fbf96-b563-4c41-b837-f0704cecfafc,20472110100000,2,1.20,2,5400119579834
332783,2024-04-30,142edab1280b2246,f0ad65cd-3d4d-4a04-b2d1-53dd637bbced,8710398502230,1,3.50,1,5410126253993
341213,2024-05-02,f8aa98ac894848ac,0ab97db6-bd06-4ca5-8fd4-c5fe28e9a5a0,5400119537643,2,1.20,2,5400113700449
339941,2024-05-02,b73fc63876e68b0b,d75ccabd-b89c-4f5f-83fd-c2e5415b200a,5400119583169,3,2.70,2,5410106095698
337044,2024-05-02,0e7ecfc90eb1dd42,ffd5bb0e-e813-4df1-b4f0-21e2aecfeb2b,2251150000000,2,1.90,2,3770018934723
345707,2024-05-03,f46feb23542f588a,1c7dcc4f-590d-4f96-9be8-7a8e8d7b0353,5400112358733,1,1.90,1,8720600061103
341821,2024-05-03,0e7ecfc90eb1dd42,1ef44333-2795-4b55-a514-dea3e6582ae5,5410221205453,1,2.80,1,8002200145491
341885,2024-05-03,0e7ecfc90eb1dd42,40abbe09-c2d4-47ec-9022-819a6c7fea84,8001100064697,1,6.60,1,5410138051051


In [4]:
correction_summary = (
    df_affected
    .groupby(
        ["sku", "old_ean", "corrected_ean"],
        dropna=False
    )
    .size()
    .reset_index(name="affected_rows")
    .sort_values("affected_rows", ascending=False)
)

display(correction_summary)

,sku,old_ean,corrected_ean,affected_rows
0,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3336374001754,8719200263796,1
1,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3487400006769,8719200263796,1
2,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3560071426620,8719200263796,1
3,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,4014071692718,8719200263796,1
4,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,5400119566766,8719200263796,1
5,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,5410138049256,8719200263796,1
6,0ab97db6-bd06-4ca5-8fd4-c5fe28e9a5a0,5400119537643,5400113700449,1
7,104844f8-1bd9-4a67-b4de-132854460dc6,7613287419095,5410146409813,1
8,12baf391-2b31-477b-a7f4-855a3e836275,8710398162045,5413883005522,1
9,13380c0c-1256-4328-b6ee-e877b71d557f,2208262000000,5411863000550,1


In [5]:
affected_skus = df_affected["sku"].unique()

df_snapshot_evidence = df_prod_snap[
    df_prod_snap["sku"].isin(affected_skus)
][["sku", "ean_code", "snapshot_date"]].copy()

display(
    df_snapshot_evidence
    .sort_values(["sku", "snapshot_date"])
)

,sku,ean_code,snapshot_date
261726,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,8719200263796,2024-04-23
265463,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,8719200263796,2024-04-24
269384,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,8719200263796,2024-04-25
272998,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,8719200263796,2024-04-26
277066,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,8719200263796,2024-04-27
...,...,...,...
319010,f0ad65cd-3d4d-4a04-b2d1-53dd637bbced,5410126253993,2024-05-13
322145,f0ad65cd-3d4d-4a04-b2d1-53dd637bbced,5410126253993,2024-05-14
325377,f0ad65cd-3d4d-4a04-b2d1-53dd637bbced,5410126253993,2024-05-15
328508,f0ad65cd-3d4d-4a04-b2d1-53dd637bbced,5410126253993,2024-05-16


In [6]:
# sku	old_ean	old_ean_exists_in_snap	old_ean_product_names	corrected_ean	corrected_ean_exists_in_snap	corrected_ean_product_names

# ============================================================
# 1. Build EAN -> product names mapping from snapshots
# ============================================================

# One EAN may have multiple product names across snapshots.
# Keep all distinct non-null names rather than arbitrarily
# choosing the first one.

ean_name_mapping = (
    df_prod_snap
    .dropna(subset=["ean_code", "product_name"])
    .groupby("ean_code")["product_name"]
    .agg(lambda names: sorted(set(names)))
    .reset_index()
)

# ============================================================
# 2. Get the product name(s) for the ORIGINAL pickup EAN
# ============================================================

df_affected_names = df_affected.merge(
    ean_name_mapping.rename(
        columns={
            "ean_code": "old_ean",
            "product_name": "old_ean_product_names"
        }
    ),
    on="old_ean",
    how="left",
    validate="many_to_one"
)

# ============================================================
# 3. Get the product name(s) for the CORRECTED snapshot EAN
# ============================================================

df_affected_names = df_affected_names.merge(
    ean_name_mapping.rename(
        columns={
            "ean_code": "corrected_ean",
            "product_name": "corrected_ean_product_names"
        }
    ),
    on="corrected_ean",
    how="left",
    validate="many_to_one"
)

# ============================================================
# 4. Indicate whether each EAN exists in snapshots
# ============================================================

snapshot_eans = set(df_prod_snap["ean_code"].dropna())

df_affected_names["old_ean_exists_in_snap"] = (
    df_affected_names["old_ean"].isin(snapshot_eans)
)

df_affected_names["corrected_ean_exists_in_snap"] = (
    df_affected_names["corrected_ean"].isin(snapshot_eans)
)

# ============================================================
# 5. Display results
# ============================================================

columns_to_display = [
    "order_date",
    "sku",
    "old_ean",
    "old_ean_exists_in_snap",
    "old_ean_product_names",
    "corrected_ean",
    "corrected_ean_exists_in_snap",
    "corrected_ean_product_names"
]

# Include pickup_point if present
if "pickup_point" in df_affected_names.columns:
    columns_to_display.insert(1, "pickup_point")

display(
    df_affected_names[columns_to_display]
    .sort_values(["order_date", "sku"])
)

,order_date,pickup_point,sku,old_ean,old_ean_exists_in_snap,old_ean_product_names,corrected_ean,corrected_ean_exists_in_snap,corrected_ean_product_names
0,2024-04-25,2542941672fd21b8,3cf2950b-da59-44ee-b5e9-d5d86aba06e5,5411761258817,True,[Filet de poulet - Aoste - 120 g (8 tranches)],5400119547581,True,[Quiche Jambon | Poireau]
2,2024-04-30,b73fc63876e68b0b,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3487400006769,True,[BenS Original Riz à la Mediterranéenne],8719200263796,True,[SOLO Margarine | Cuire et rôtir | Barquette |...
3,2024-04-30,cbcb62162e461101,d36fbf96-b563-4c41-b837-f0704cecfafc,20472110100000,True,[Carbonnades 1er Choix],5400119579834,True,[Americain Préparé | Martino]
1,2024-04-30,142edab1280b2246,f0ad65cd-3d4d-4a04-b2d1-53dd637bbced,8710398502230,True,[Lays grills fumé],5410126253993,True,[Lotus oeufs de Pâques MIX lait blanc crème sp...
6,2024-05-02,f8aa98ac894848ac,0ab97db6-bd06-4ca5-8fd4-c5fe28e9a5a0,5400119537643,True,[Mozzarella | Râpée],5400113700449,True,[mini sandwiches au beurre]
5,2024-05-02,b73fc63876e68b0b,d75ccabd-b89c-4f5f-83fd-c2e5415b200a,5400119583169,True,[Chili con carne | Riz mexicain | Nachos],5410106095698,True,[Bio Toasts Pois Chiches Goût D’épices Marocai...
4,2024-05-02,0e7ecfc90eb1dd42,ffd5bb0e-e813-4df1-b4f0-21e2aecfeb2b,2251150000000,True,[Americain nature],3770018934723,True,[2 Hachés burger goût fumé végétal | Accro]
11,2024-05-03,f46feb23542f588a,1c7dcc4f-590d-4f96-9be8-7a8e8d7b0353,5400112358733,True,[Carottes rapées - Bio Delhaize],8720600061103,True,[Petits pois & jeunes carottes extra fins]
7,2024-05-03,0e7ecfc90eb1dd42,1ef44333-2795-4b55-a514-dea3e6582ae5,5410221205453,True,[Schweppes Virgin Mojito Zéro 6x33cl],8002200145491,True,[Nescafe Capsules x16 compatibles Dolce Gusto ...
8,2024-05-03,0e7ecfc90eb1dd42,40abbe09-c2d4-47ec-9022-819a6c7fea84,8001100064697,True,[Torta Torte in Festa con panna e cioccolato B...,5410138051051,True,[Café Capsules x40 Dessert Lungo | Intensité 0...


In [7]:
ean_sku_mapping = (
    df_prod_snap
    .dropna(subset=["ean_code", "sku"])
    .groupby("ean_code")["sku"]
    .agg(lambda skus: sorted(set(skus)))
    .reset_index()
    .rename(columns={
        "ean_code": "old_ean",
        "sku": "old_ean_snapshot_skus"
    })
)

In [8]:
df_affected_names = df_affected_names.merge(
    ean_sku_mapping,
    on="old_ean",
    how="left",
    validate="many_to_one"
)

In [9]:
columns_to_display = [
    "order_date",
    "sku",                         # Original pickup SKU
    "old_ean",
    "old_ean_exists_in_snap",
    "old_ean_snapshot_skus",      # SKU(s) associated with old EAN in snapshots
    "old_ean_product_names",
    "corrected_ean",
    "corrected_ean_exists_in_snap",
    "corrected_ean_product_names"
]

if "pickup_point" in df_affected_names.columns:
    columns_to_display.insert(1, "pickup_point")

display(
    df_affected_names[columns_to_display]
    .sort_values(["order_date", "sku"])
)

,order_date,pickup_point,sku,old_ean,old_ean_exists_in_snap,old_ean_snapshot_skus,old_ean_product_names,corrected_ean,corrected_ean_exists_in_snap,corrected_ean_product_names
0,2024-04-25,2542941672fd21b8,3cf2950b-da59-44ee-b5e9-d5d86aba06e5,5411761258817,True,"[01056d81-bfa8-45db-b802-99fdc135f083, 0126c79...",[Filet de poulet - Aoste - 120 g (8 tranches)],5400119547581,True,[Quiche Jambon | Poireau]
2,2024-04-30,b73fc63876e68b0b,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3487400006769,True,"[092b9e44-6902-4181-9032-74a230f00d8c, 137ce18...",[BenS Original Riz à la Mediterranéenne],8719200263796,True,[SOLO Margarine | Cuire et rôtir | Barquette |...
3,2024-04-30,cbcb62162e461101,d36fbf96-b563-4c41-b837-f0704cecfafc,20472110100000,True,"[04c820e3-e96e-4d9d-9e51-59b59827be03, 1527f9d...",[Carbonnades 1er Choix],5400119579834,True,[Americain Préparé | Martino]
1,2024-04-30,142edab1280b2246,f0ad65cd-3d4d-4a04-b2d1-53dd637bbced,8710398502230,True,"[06e1cc21-1f5c-405c-b912-a16fe235307a, 08a4e95...",[Lays grills fumé],5410126253993,True,[Lotus oeufs de Pâques MIX lait blanc crème sp...
6,2024-05-02,f8aa98ac894848ac,0ab97db6-bd06-4ca5-8fd4-c5fe28e9a5a0,5400119537643,True,"[107304cc-5909-4c12-be6f-bd97d3b50a5a, 10f3e56...",[Mozzarella | Râpée],5400113700449,True,[mini sandwiches au beurre]
5,2024-05-02,b73fc63876e68b0b,d75ccabd-b89c-4f5f-83fd-c2e5415b200a,5400119583169,True,"[002dccc1-9157-4d43-8ae6-d44f357ceb99, 026fee9...",[Chili con carne | Riz mexicain | Nachos],5410106095698,True,[Bio Toasts Pois Chiches Goût D’épices Marocai...
4,2024-05-02,0e7ecfc90eb1dd42,ffd5bb0e-e813-4df1-b4f0-21e2aecfeb2b,2251150000000,True,"[016eff64-2f69-48fd-b2c5-9e8579c4612d, 04f14ea...",[Americain nature],3770018934723,True,[2 Hachés burger goût fumé végétal | Accro]
11,2024-05-03,f46feb23542f588a,1c7dcc4f-590d-4f96-9be8-7a8e8d7b0353,5400112358733,True,"[000c5ee1-2505-46c5-8cf5-e7db22609914, 003724c...",[Carottes rapées - Bio Delhaize],8720600061103,True,[Petits pois & jeunes carottes extra fins]
7,2024-05-03,0e7ecfc90eb1dd42,1ef44333-2795-4b55-a514-dea3e6582ae5,5410221205453,True,"[02d344be-d297-4ce1-8775-c619ce76ec75, 077499a...",[Schweppes Virgin Mojito Zéro 6x33cl],8002200145491,True,[Nescafe Capsules x16 compatibles Dolce Gusto ...
8,2024-05-03,0e7ecfc90eb1dd42,40abbe09-c2d4-47ec-9022-819a6c7fea84,8001100064697,True,"[4d80e06a-06c8-4139-8dfb-db4693b26ec9, 5570b83...",[Torta Torte in Festa con panna e cioccolato B...,5410138051051,True,[Café Capsules x40 Dessert Lungo | Intensité 0...


In [10]:
df_affected_names["old_ean_matches_pickup_sku"] = (
    df_affected_names.apply(
        lambda row: (
            row["sku"] in row["old_ean_snapshot_skus"]
            if isinstance(row["old_ean_snapshot_skus"], list)
            else False
        ),
        axis=1
    )
)

display(
    df_affected_names[
        [
            "sku",
            "old_ean",
            "old_ean_snapshot_skus",
            "old_ean_matches_pickup_sku",
            "old_ean_product_names",
            "corrected_ean",
            "corrected_ean_product_names"
        ]
    ]
)

,sku,old_ean,old_ean_snapshot_skus,old_ean_matches_pickup_sku,old_ean_product_names,corrected_ean,corrected_ean_product_names
0,3cf2950b-da59-44ee-b5e9-d5d86aba06e5,5411761258817,"[01056d81-bfa8-45db-b802-99fdc135f083, 0126c79...",False,[Filet de poulet - Aoste - 120 g (8 tranches)],5400119547581,[Quiche Jambon | Poireau]
1,f0ad65cd-3d4d-4a04-b2d1-53dd637bbced,8710398502230,"[06e1cc21-1f5c-405c-b912-a16fe235307a, 08a4e95...",False,[Lays grills fumé],5410126253993,[Lotus oeufs de Pâques MIX lait blanc crème sp...
2,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,3487400006769,"[092b9e44-6902-4181-9032-74a230f00d8c, 137ce18...",False,[BenS Original Riz à la Mediterranéenne],8719200263796,[SOLO Margarine | Cuire et rôtir | Barquette |...
3,d36fbf96-b563-4c41-b837-f0704cecfafc,20472110100000,"[04c820e3-e96e-4d9d-9e51-59b59827be03, 1527f9d...",False,[Carbonnades 1er Choix],5400119579834,[Americain Préparé | Martino]
4,ffd5bb0e-e813-4df1-b4f0-21e2aecfeb2b,2251150000000,"[016eff64-2f69-48fd-b2c5-9e8579c4612d, 04f14ea...",False,[Americain nature],3770018934723,[2 Hachés burger goût fumé végétal | Accro]
5,d75ccabd-b89c-4f5f-83fd-c2e5415b200a,5400119583169,"[002dccc1-9157-4d43-8ae6-d44f357ceb99, 026fee9...",False,[Chili con carne | Riz mexicain | Nachos],5410106095698,[Bio Toasts Pois Chiches Goût D’épices Marocai...
6,0ab97db6-bd06-4ca5-8fd4-c5fe28e9a5a0,5400119537643,"[107304cc-5909-4c12-be6f-bd97d3b50a5a, 10f3e56...",False,[Mozzarella | Râpée],5400113700449,[mini sandwiches au beurre]
7,1ef44333-2795-4b55-a514-dea3e6582ae5,5410221205453,"[02d344be-d297-4ce1-8775-c619ce76ec75, 077499a...",False,[Schweppes Virgin Mojito Zéro 6x33cl],8002200145491,[Nescafe Capsules x16 compatibles Dolce Gusto ...
8,40abbe09-c2d4-47ec-9022-819a6c7fea84,8001100064697,"[4d80e06a-06c8-4139-8dfb-db4693b26ec9, 5570b83...",False,[Torta Torte in Festa con panna e cioccolato B...,5410138051051,[Café Capsules x40 Dessert Lungo | Intensité 0...
9,ea6bbe25-87d9-452c-8b56-73b15ed88073,5411901092202,"[8a294f20-eac5-4b9f-bb68-093e112e67d3, a9a5966...",False,[Chocolat | Deli Dino | Lait couleur],5400119573375,[Albondigas]


In [15]:
import pandas as pd

# Get distinct, non-null SKU-EAN pairs
pickup_pairs = (
    df_pickup[["sku", "ean_code"]]
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
)

snap_pairs = (
    df_prod_snap[["sku", "ean_code"]]
    .dropna(subset=["sku", "ean_code"])
    .drop_duplicates()
)

# Count distinct EANs per SKU
pickup_ean_counts = pickup_pairs.groupby("sku")["ean_code"].nunique()
snap_ean_counts = snap_pairs.groupby("sku")["ean_code"].nunique()

# Identify SKUs with multiple EANs
multi_ean_skus_pickup = pickup_ean_counts[
    pickup_ean_counts > 1
].index

multi_ean_skus_snap = snap_ean_counts[
    snap_ean_counts > 1
].index

# Display each SKU with its list of EANs
df_multi_ean_pickup = (
    pickup_pairs[
        pickup_pairs["sku"].isin(multi_ean_skus_pickup)
    ]
    .groupby("sku")["ean_code"]
    .agg(list)
    .reset_index(name="ean_codes")
)

df_multi_ean_snap = (
    snap_pairs[
        snap_pairs["sku"].isin(multi_ean_skus_snap)
    ]
    .groupby("sku")["ean_code"]
    .agg(list)
    .reset_index(name="ean_codes")
)

print(f"SKUs with multiple EANs in pickup: {len(df_multi_ean_pickup)}")
print(f"SKUs with multiple EANs in snapshots: {len(df_multi_ean_snap)}")

display(df_multi_ean_pickup)
display(df_multi_ean_snap)

SKUs with multiple EANs in pickup: 35
SKUs with multiple EANs in snapshots: 11


,sku,ean_codes
0,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,"[8719200263796, 3487400006769, 5400119566766, ..."
1,0a7d450b-5bab-4cc1-911a-b034ad2dbcca,"[871039852411900100000, 871039852411900100]"
2,0ab97db6-bd06-4ca5-8fd4-c5fe28e9a5a0,"[5400113700449, 5400119537643]"
3,104844f8-1bd9-4a67-b4de-132854460dc6,"[7613287419095, 5410146409813]"
4,12baf391-2b31-477b-a7f4-855a3e836275,"[5413883005522, 8710398162045]"
5,140ed542-5cb9-4110-aad5-f2036df614ff,"[5400119584227, 2250491000000]"
6,15d19704-0ecf-4eb7-9e76-7a3245a16d8d,"[87365368, 5400601061618]"
7,1c7dcc4f-590d-4f96-9be8-7a8e8d7b0353,"[8720600061103, 5400112358733]"
8,1ef44333-2795-4b55-a514-dea3e6582ae5,"[8002200145491, 5410221205453, 5425020166423]"
9,1fb27241-c5b0-4334-a2ef-b54987640af9,"[5410228215066, 5400119568746]"


,sku,ean_codes
0,24542b78-1b79-4863-bdce-4da9eafcc638,"[67000012860, 067000012860]"
1,27feb930-2213-4bd2-9962-13662ad94a25,"[79581184010, 079581184010]"
2,3e6398fa-b66d-4eb6-aeb1-2932aae017d8,"[24000011859, 024000011859]"
3,5351c7a9-96e2-498a-9efa-d4d804cac62f,"[24000040095, 024000040095]"
4,cd542a42-b55c-4ce5-8006-75a730247bb2,"[24000197409, 024000197409]"
5,d6aeb5ca-43b3-4c75-8f24-7cfcfe768373,"[24000025528, 024000025528]"
6,de6cee29-8de3-4964-bbc6-daa4251605e1,"[41420078477, 041420078477]"
7,e12c96af-2b2e-4e74-aae0-c20b8f1263e4,"[70177176914, 070177176914]"
8,e58bdae4-0fdc-453f-9e16-f6608bb960d7,"[71570017194, 071570017194]"
9,f69a3f7c-fe78-47e2-871c-4e7217709a02,"[67000012860, 067000012860]"


In [16]:
#Snapshots and pickup multiple ean per sku and 
#number of records per sku and date range for each sku in pickup and snapshots

# ============================================================
# 1. Prepare dates
# ============================================================

df_pickup_review = df_pickup.copy()
df_snap_review = df_prod_snap.copy()

df_pickup_review["order_date"] = pd.to_datetime(
    df_pickup_review["order_date"], errors="coerce"
)

df_snap_review["snapshot_date"] = pd.to_datetime(
    df_snap_review["snapshot_date"], errors="coerce"
)


# ============================================================
# 2. Pickup: Date range and record count per multi-EAN SKU
# ============================================================

df_pickup_multi_records = df_pickup_review[
    df_pickup_review["sku"].isin(multi_ean_skus_pickup)
].copy()

df_multi_ean_pickup_summary = (
    df_pickup_multi_records
    .groupby("sku")
    .agg(
        min_date=("order_date", "min"),
        max_date=("order_date", "max"),
        total_records=("sku", "size"),
        unique_eans=("ean_code", "nunique")
    )
    .reset_index()
)

# Attach list of EANs from your existing analysis
df_multi_ean_pickup_summary = df_multi_ean_pickup_summary.merge(
    df_multi_ean_pickup,
    on="sku",
    how="left",
    validate="one_to_one"
)

df_multi_ean_pickup_summary = (
    df_multi_ean_pickup_summary
    .sort_values("total_records", ascending=False)
)


# ============================================================
# 3. Snapshots: Date range and record count per multi-EAN SKU
# ============================================================

df_snap_multi_records = df_snap_review[
    df_snap_review["sku"].isin(multi_ean_skus_snap)
].copy()

df_multi_ean_snap_summary = (
    df_snap_multi_records
    .groupby("sku")
    .agg(
        min_date=("snapshot_date", "min"),
        max_date=("snapshot_date", "max"),
        total_records=("sku", "size"),
        unique_eans=("ean_code", "nunique")
    )
    .reset_index()
)

df_multi_ean_snap_summary = df_multi_ean_snap_summary.merge(
    df_multi_ean_snap,
    on="sku",
    how="left",
    validate="one_to_one"
)

df_multi_ean_snap_summary = (
    df_multi_ean_snap_summary
    .sort_values("total_records", ascending=False)
)


# ============================================================
# 4. Print overall statistics
# ============================================================

print("=== PICKUP MULTI-EAN ANALYSIS ===")

print(f"SKUs with multiple EANs: {len(multi_ean_skus_pickup)}")

print(
    f"Total pickup records potentially requiring review: "
    f"{len(df_pickup_multi_records):,}"
)

print(
    f"Earliest order date: "
    f"{df_pickup_multi_records['order_date'].min()}"
)

print(
    f"Latest order date: "
    f"{df_pickup_multi_records['order_date'].max()}"
)


print("\n=== SNAPSHOT MULTI-EAN ANALYSIS ===")

print(f"SKUs with multiple EANs: {len(multi_ean_skus_snap)}")

print(
    f"Total snapshot records potentially requiring review: "
    f"{len(df_snap_multi_records):,}"
)

print(
    f"Earliest snapshot date: "
    f"{df_snap_multi_records['snapshot_date'].min()}"
)

print(
    f"Latest snapshot date: "
    f"{df_snap_multi_records['snapshot_date'].max()}"
)


# ============================================================
# 5. Display detailed results
# ============================================================

print("\n=== PICKUP: MULTI-EAN SKUs ===")
display(df_multi_ean_pickup_summary)

print("\n=== SNAPSHOTS: MULTI-EAN SKUs ===")
display(df_multi_ean_snap_summary)

=== PICKUP MULTI-EAN ANALYSIS ===
SKUs with multiple EANs: 35
Total pickup records potentially requiring review: 1,525
Earliest order date: 2024-02-23 00:00:00
Latest order date: 2024-07-27 00:00:00

=== SNAPSHOT MULTI-EAN ANALYSIS ===
SKUs with multiple EANs: 11
Total snapshot records potentially requiring review: 490
Earliest snapshot date: 2024-09-18 00:00:00
Latest snapshot date: 2025-01-06 00:00:00

=== PICKUP: MULTI-EAN SKUs ===


,sku,min_date,max_date,total_records,unique_eans,ean_codes
6,15d19704-0ecf-4eb7-9e76-7a3245a16d8d,2024-02-23,2024-05-21,410,2,"[87365368, 5400601061618]"
0,0a43b4ec-d8db-4679-9b50-2e6c869a4d2e,2024-04-23,2024-05-11,162,7,"[8719200263796, 3487400006769, 5400119566766, ..."
8,1ef44333-2795-4b55-a514-dea3e6582ae5,2024-04-24,2024-06-28,154,3,"[8002200145491, 5410221205453, 5425020166423]"
26,d6a34af7-bc99-4efc-8e69-a82869b415e3,2024-05-04,2024-07-27,151,2,"[4820078570979, 5413110052367]"
13,40abbe09-c2d4-47ec-9022-819a6c7fea84,2024-04-23,2024-06-03,93,2,"[5410138051051, 8001100064697]"
31,e6c43336-c92c-4588-bfdc-db1184242add,2024-05-03,2024-05-10,87,3,"[3023290047941, 5400113594291, 3228170823404]"
25,c41703dd-e648-4462-a933-3cda0d0000a1,2024-02-29,2024-05-24,71,2,"[8719500005690, 3250392615536]"
14,5a0fc00e-11ee-4adb-9201-3684848e5b85,2024-05-13,2024-05-18,66,2,"[9998883, 5420036416210]"
23,affd56f3-0a66-4ac1-a07a-82ff0e73c56c,2024-07-18,2024-07-20,37,2,"[5400119589635, 5038862004558]"
9,1fb27241-c5b0-4334-a2ef-b54987640af9,2024-05-23,2024-05-31,34,2,"[5410228215066, 5400119568746]"



=== SNAPSHOTS: MULTI-EAN SKUs ===


,sku,min_date,max_date,total_records,unique_eans,ean_codes
1,27feb930-2213-4bd2-9962-13662ad94a25,2024-09-25,2025-01-06,104,2,"[79581184010, 079581184010]"
6,de6cee29-8de3-4964-bbc6-daa4251605e1,2024-09-18,2024-12-05,79,2,"[41420078477, 041420078477]"
4,cd542a42-b55c-4ce5-8006-75a730247bb2,2024-10-29,2025-01-04,68,2,"[24000197409, 024000197409]"
8,e58bdae4-0fdc-453f-9e16-f6608bb960d7,2024-09-18,2024-11-19,63,2,"[71570017194, 071570017194]"
9,f69a3f7c-fe78-47e2-871c-4e7217709a02,2024-11-15,2024-12-25,41,2,"[67000012860, 067000012860]"
5,d6aeb5ca-43b3-4c75-8f24-7cfcfe768373,2024-11-13,2024-12-22,40,2,"[24000025528, 024000025528]"
3,5351c7a9-96e2-498a-9efa-d4d804cac62f,2024-11-13,2024-12-19,37,2,"[24000040095, 024000040095]"
0,24542b78-1b79-4863-bdce-4da9eafcc638,2024-11-15,2024-12-12,28,2,"[67000012860, 067000012860]"
2,3e6398fa-b66d-4eb6-aeb1-2932aae017d8,2024-11-13,2024-11-25,13,2,"[24000011859, 024000011859]"
7,e12c96af-2b2e-4e74-aae0-c20b8f1263e4,2024-11-13,2024-11-22,10,2,"[70177176914, 070177176914]"


In [14]:
# SKUs with matching EANs: 1303461
# SKUs with different EANs: 130

# ============================================================
# 1. Exclude SKUs with multiple EANs in either dataset
# ============================================================

ambiguous_skus = (
    set(multi_ean_skus_pickup)
    | set(multi_ean_skus_snap)
)

pickup_clean = pickup_pairs[
    ~pickup_pairs["sku"].isin(ambiguous_skus)
].copy()

snap_clean = snap_pairs[
    ~snap_pairs["sku"].isin(ambiguous_skus)
].copy()


# ============================================================
# 2. Compare EANs for SKUs present in both datasets
# ============================================================

df_sku_ean_comparison = pickup_clean.merge(
    snap_clean,
    on="sku",
    how="inner",
    suffixes=("_pickup", "_snap"),
    validate="one_to_one"
)


# ============================================================
# 3. Identify different EAN assignments
# ============================================================

df_sku_ean_differences = df_sku_ean_comparison[
    df_sku_ean_comparison["ean_code_pickup"]
    != df_sku_ean_comparison["ean_code_snap"]
].copy()

df_sku_ean_matches = df_sku_ean_comparison[
    df_sku_ean_comparison["ean_code_pickup"]
    == df_sku_ean_comparison["ean_code_snap"]
].copy()


# ============================================================
# 4. Display summary
# ============================================================

print("=== SKU-EAN COMPARISON ===")
print(f"SKUs excluded due to multiple EANs: {len(ambiguous_skus)}")
print(f"SKUs compared in both datasets: {len(df_sku_ean_comparison)}")
print(f"SKUs with matching EANs: {len(df_sku_ean_matches)}")
print(f"SKUs with different EANs: {len(df_sku_ean_differences)}")

display(
    df_sku_ean_differences.sort_values("sku")
)

=== SKU-EAN COMPARISON ===
SKUs excluded due to multiple EANs: 46
SKUs compared in both datasets: 1303591
SKUs with matching EANs: 1303461
SKUs with different EANs: 130


,sku,ean_code_pickup,ean_code_snap
348801,0023c4f2-cc0e-4094-8ac0-d647f3c0273d,024000017677,24000017677
266346,010a9eb8-8427-4948-b986-dc66a606487b,038900033125,38900033125
382585,025b0474-ee07-4d0d-80d2-f0b3f18765fd,031200456991,31200456991
269989,032883ef-62e6-42e3-993b-b0fb5eed68bb,031200456991,31200456991
357108,05bf4207-2c6e-40b0-9c46-4138b3f2c7ea,038900029746,38900029746
...,...,...,...
322098,f9c4e2d1-0afb-411f-9c8f-389286d8c61b,024000106654,24000106654
873568,fb84e253-caa6-4e94-b9ad-4a91bf33ad31,87148111754830,8714811175483
322895,fc1e656e-4082-405b-b66c-ed68beb00a2a,024000197409,24000197409
264629,fcc9a936-ed1f-406c-8c5e-abe7fcdec6e0,038900033125,38900033125
